# A grokking regime where minibatch noise is dynamically relevant

This notebook uses the same modular-addition toy model as the earlier grokking experiment, but reduces coupled L2 decay from $10^{-4}$ to **$7\times10^{-5}$**. In this regime, stochasticity changes whether the model groks within the 12,000-update budget:

- **batch 1 (high noise):** groks reliably;
- **batch 32 (lower noise):** does not reach 90% held-out accuracy;
- **full batch (deterministic):** does not reach 90% held-out accuracy.

The data split, initialization, learning rate, objective, decay, and optimizer-update count are fixed. A minibatch mean is an unbiased estimate of the full gradient, so changing batch size changes gradient variance while preserving expected per-update drift. This is an update-matched comparison; the number of examples processed is reported separately.

Seven paired sampler seeds test whether the effect is robust. A representative seed retains 1,000 raw post-update parameter checkpoints and exact trailing averages over the preceding 50 **actual** updates. PCA is fit once to the deterministic averaged path and reused for every condition, making coordinates directly comparable.

The full-space residual is lag corrected: a trailing 50-step mean is centered around $t-24.5$, so it is compared with the raw checkpoint at approximately $t-24$, rather than incorrectly comparing it with $w_t$. This distinction prevents smooth drift from being mislabeled as noise.

For background on SGD noise strength and its dependence on learning rate and minibatch size, see [Ziyin et al. (2021)](https://arxiv.org/abs/2102.05375).


In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

SEED = 2026
SAMPLER_SEEDS = tuple(range(7))
REPRESENTATIVE_SEED = 1
MODULUS = 23
HIDDEN_WIDTH = 128
LEARNING_RATE = 3.0
WEIGHT_DECAY = 7e-5
TOTAL_UPDATES = 12_000
N_CHECKPOINTS = 1_000
CHECKPOINT_STRIDE = TOTAL_UPDATES // N_CHECKPOINTS
AVERAGE_WINDOW = 50
MAX_STOCHASTIC_BATCH = 32
SUSTAINED_CHECKPOINTS = 5
OUT_DIR = Path("diff_plots_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_FULL = "Full batch"
LABEL_B32 = "Batch 32"
LABEL_B1 = "Batch 1"
CONDITION_ORDER = (LABEL_FULL, LABEL_B32, LABEL_B1)
COLORS = {
    LABEL_FULL: "#777777",
    LABEL_B32: "#4C78A8",
    LABEL_B1: "#F58518",
}

assert CHECKPOINT_STRIDE == 12
assert N_CHECKPOINTS * CHECKPOINT_STRIDE == TOTAL_UPDATES

try:
    gpu_devices = jax.devices("gpu")
except RuntimeError as exc:
    raise RuntimeError("This notebook requires a CUDA-enabled JAX installation.") from exc
if len(gpu_devices) <= 7:
    raise RuntimeError(
        f"cuda:7 was requested, but only {len(gpu_devices)} GPU device(s) are visible."
    )
DEVICE = gpu_devices[7]
if DEVICE.id != 7:
    raise RuntimeError(f"Expected physical cuda:7, selected {DEVICE}.")

print(f"JAX {jax.__version__}")
print(f"Using {DEVICE} ({DEVICE.device_kind})")
print(f"Plots will be written to {OUT_DIR.resolve()}")


## Fixed task, split, model, and optimizer

The task is modular addition mod 23. Train/test assignment is by unordered-pair orbit, so $(a,b)$ and $(b,a)$ always remain in the same split. The model is a no-bias quadratic network with 8,832 parameters, trained with genuine minibatch SGD and coupled L2 decay:

$$\theta \leftarrow \theta - 3\left(\nabla L_{\mathrm{batch}} + 7\times10^{-5}\theta\right).$$

There is no momentum, clipping, or adaptive optimizer state.


In [ ]:
# Transpose-safe split on the host.
rng = np.random.default_rng(SEED)
upper_a, upper_b = np.triu_indices(MODULUS)
group_order = rng.permutation(upper_a.size)
selected_groups = group_order[: upper_a.size // 2]
train_matrix = np.zeros((MODULUS, MODULUS), dtype=bool)
train_matrix[upper_a[selected_groups], upper_b[selected_groups]] = True
train_matrix |= train_matrix.T
assert np.array_equal(train_matrix, train_matrix.T)

operand_a, operand_b = np.meshgrid(
    np.arange(MODULUS), np.arange(MODULUS), indexing="ij"
)
all_pairs_np = np.stack([operand_a.ravel(), operand_b.ravel()], axis=1)
all_targets_np = (all_pairs_np[:, 0] + all_pairs_np[:, 1]) % MODULUS
train_mask_np = train_matrix.ravel()
train_pairs_np = all_pairs_np[train_mask_np]
train_targets_np = all_targets_np[train_mask_np]
test_pairs_np = all_pairs_np[~train_mask_np]
test_targets_np = all_targets_np[~train_mask_np]

for residue in range(MODULUS):
    assert np.any(train_targets_np == residue)
    assert np.any(test_targets_np == residue)


def init_params(key):
    key_in, key_out = jax.random.split(key)
    input_limit = 1.0 / np.sqrt(2 * MODULUS)
    output_limit = 1.0 / np.sqrt(HIDDEN_WIDTH)
    w_input = jax.random.uniform(
        key_in,
        (2 * MODULUS, HIDDEN_WIDTH),
        minval=-input_limit,
        maxval=input_limit,
        dtype=jnp.float32,
    )
    w_output = jax.random.uniform(
        key_out,
        (HIDDEN_WIDTH, MODULUS),
        minval=-output_limit,
        maxval=output_limit,
        dtype=jnp.float32,
    )
    return (w_input, w_output)


def model_logits(params, pairs):
    w_input, w_output = params
    left = jax.nn.one_hot(pairs[:, 0], MODULUS) @ w_input[:MODULUS]
    right = jax.nn.one_hot(pairs[:, 1], MODULUS) @ w_input[MODULUS:]
    return (left + right) ** 2 @ w_output


def mse_loss(params, pairs, targets):
    predictions = model_logits(params, pairs)
    one_hot_targets = jax.nn.one_hot(targets, MODULUS)
    return jnp.mean((predictions - one_hot_targets) ** 2)


def flatten_params(params):
    return jnp.concatenate([parameter.reshape(-1) for parameter in params])


def unflatten_params(vector):
    input_size = 2 * MODULUS * HIDDEN_WIDTH
    w_input = vector[:input_size].reshape(2 * MODULUS, HIDDEN_WIDTH)
    w_output = vector[input_size:].reshape(HIDDEN_WIDTH, MODULUS)
    return (w_input, w_output)


with jax.default_device(DEVICE):
    train_pairs = jnp.asarray(train_pairs_np, dtype=jnp.int32)
    train_targets = jnp.asarray(train_targets_np, dtype=jnp.int32)
    test_pairs = jnp.asarray(test_pairs_np, dtype=jnp.int32)
    test_targets = jnp.asarray(test_targets_np, dtype=jnp.int32)

    root_key = jax.random.key(SEED)
    init_key, base_sampler_key = jax.random.split(root_key)
    initial_params = init_params(init_key)

parameter_count = int(flatten_params(initial_params).size)
assert parameter_count == 8_832
assert all(parameter.devices() == {DEVICE} for parameter in initial_params)
assert train_pairs.devices() == {DEVICE}

print(f"Train pairs: {train_pairs.shape[0]}; held-out pairs: {test_pairs.shape[0]}")
print(f"Parameters: {parameter_count:,}; chance accuracy: {100 / MODULUS:.2f}%")
for batch_size in (1, 32, train_pairs.shape[0]):
    print(
        f"Batch {batch_size:>3}: {TOTAL_UPDATES * batch_size:,} examples "
        f"processed in {TOTAL_UPDATES:,} updates"
    )


## Training and exact trajectory recording

All metrics are evaluated on the complete train and held-out splits every 12 updates. For the representative sampler seed, a 50-vector GPU ring buffer is updated after every optimizer step, retaining exactly 1,000 raw checkpoints and their exact trailing means. The remaining seeds run the same optimization but retain metrics only.


In [ ]:
def make_schedule(sampler_seed):
    key = jax.random.fold_in(base_sampler_key, sampler_seed)
    return jax.random.randint(
        key,
        shape=(TOTAL_UPDATES, MAX_STOCHASTIC_BATCH),
        minval=0,
        maxval=train_pairs.shape[0],
        dtype=jnp.int32,
    )


def full_batch_schedule():
    indices = jnp.arange(train_pairs.shape[0], dtype=jnp.int32)
    return jnp.broadcast_to(indices, (TOTAL_UPDATES, train_pairs.shape[0]))


def evaluate_params(params, all_train_pairs, all_train_targets, all_test_pairs, all_test_targets):
    train_loss = mse_loss(params, all_train_pairs, all_train_targets)
    test_loss = mse_loss(params, all_test_pairs, all_test_targets)
    train_accuracy = jnp.mean(
        jnp.argmax(model_logits(params, all_train_pairs), axis=1) == all_train_targets
    )
    test_accuracy = jnp.mean(
        jnp.argmax(model_logits(params, all_test_pairs), axis=1) == all_test_targets
    )
    parameter_norm = jnp.linalg.norm(flatten_params(params))
    return train_loss, test_loss, train_accuracy, test_accuracy, parameter_norm


@jax.jit(device=DEVICE)
def run_with_path(
    params,
    all_train_pairs,
    all_train_targets,
    all_test_pairs,
    all_test_targets,
    schedule,
):
    n_params = flatten_params(params).size
    history_buffer = jnp.zeros((AVERAGE_WINDOW, n_params), dtype=jnp.float32)
    rolling_sum = jnp.zeros((n_params,), dtype=jnp.float32)
    update_number = jnp.asarray(0, dtype=jnp.int32)

    def checkpoint_block(carry, block_indices):
        def update_once(inner_carry, indices):
            current_params, buffer, current_sum, current_update = inner_carry
            _, gradients = jax.value_and_grad(mse_loss)(
                current_params,
                all_train_pairs[indices],
                all_train_targets[indices],
            )
            next_params = jax.tree_util.tree_map(
                lambda parameter, gradient: parameter
                - LEARNING_RATE * (gradient + WEIGHT_DECAY * parameter),
                current_params,
                gradients,
            )
            current_update = current_update + 1
            vector = flatten_params(next_params)
            slot = (current_update - 1) % AVERAGE_WINDOW
            outgoing = buffer[slot]
            buffer = buffer.at[slot].set(vector)
            current_sum = current_sum + vector - jnp.where(
                current_update > AVERAGE_WINDOW,
                outgoing,
                jnp.zeros_like(outgoing),
            )
            return (next_params, buffer, current_sum, current_update), None

        carry, _ = jax.lax.scan(update_once, carry, block_indices)
        current_params, buffer, current_sum, current_update = carry
        raw_vector = flatten_params(current_params)
        count = jnp.minimum(current_update, AVERAGE_WINDOW).astype(jnp.float32)
        average_vector = current_sum / count
        metrics = evaluate_params(
            current_params,
            all_train_pairs,
            all_train_targets,
            all_test_pairs,
            all_test_targets,
        )
        return carry, (raw_vector, average_vector, *metrics)

    initial_carry = (
        params,
        history_buffer,
        rolling_sum,
        update_number,
    )
    blocked_schedule = schedule.reshape(
        N_CHECKPOINTS, CHECKPOINT_STRIDE, schedule.shape[-1]
    )
    return jax.lax.scan(checkpoint_block, initial_carry, blocked_schedule)


@jax.jit(device=DEVICE)
def run_metrics_only(
    params,
    all_train_pairs,
    all_train_targets,
    all_test_pairs,
    all_test_targets,
    schedule,
):
    def checkpoint_block(current_params, block_indices):
        def update_once(params_before_update, indices):
            _, gradients = jax.value_and_grad(mse_loss)(
                params_before_update,
                all_train_pairs[indices],
                all_train_targets[indices],
            )
            params_after_update = jax.tree_util.tree_map(
                lambda parameter, gradient: parameter
                - LEARNING_RATE * (gradient + WEIGHT_DECAY * parameter),
                params_before_update,
                gradients,
            )
            return params_after_update, None

        current_params, _ = jax.lax.scan(update_once, current_params, block_indices)
        metrics = evaluate_params(
            current_params,
            all_train_pairs,
            all_train_targets,
            all_test_pairs,
            all_test_targets,
        )
        return current_params, metrics

    blocked_schedule = schedule.reshape(
        N_CHECKPOINTS, CHECKPOINT_STRIDE, schedule.shape[-1]
    )
    return jax.lax.scan(checkpoint_block, params, blocked_schedule)


In [ ]:
with jax.default_device(DEVICE):
    representative_master = make_schedule(REPRESENTATIVE_SEED)
    schedules = {
        LABEL_B1: representative_master[:, :1],
        LABEL_B32: representative_master,
        LABEL_FULL: full_batch_schedule(),
    }

recorded_runs = {}
for label in CONDITION_ORDER:
    final_carry, outputs = run_with_path(
        initial_params,
        train_pairs,
        train_targets,
        test_pairs,
        test_targets,
        schedules[label],
    )
    outputs[1].block_until_ready()
    final_update = int(final_carry[-1])
    assert final_update == TOTAL_UPDATES
    assert outputs[0].shape == outputs[1].shape == (N_CHECKPOINTS, parameter_count)
    assert outputs[0].devices() == {DEVICE}
    recorded_runs[label] = outputs
    print(
        f"{label:>10}: final train={100 * float(outputs[4][-1]):6.2f}%, "
        f"test={100 * float(outputs[5][-1]):6.2f}%"
    )

checkpoint_steps = CHECKPOINT_STRIDE * np.arange(1, N_CHECKPOINTS + 1)


## Seven paired sampler seeds

“Grokking” is the first checkpoint beginning a five-checkpoint streak at or above 90% held-out accuracy. A missing hitting time is right-censored at 12,000 updates. All stochastic runs share the same initialization; only their sampler key differs.


In [ ]:
def first_sustained_index(values, threshold, length=SUSTAINED_CHECKPOINTS):
    hits = (values >= threshold).astype(np.int32)
    run_lengths = np.convolve(hits, np.ones(length, dtype=np.int32), mode="valid")
    candidates = np.flatnonzero(run_lengths == length)
    return int(candidates[0]) if candidates.size else None


def host_metrics(path_outputs):
    return tuple(np.asarray(jax.device_get(array)) for array in path_outputs[2:])


def summarize_metrics(metrics):
    train_loss, test_loss, train_accuracy, test_accuracy, parameter_norm = metrics
    train_index = first_sustained_index(train_accuracy, 0.99)
    test_index = first_sustained_index(test_accuracy, 0.90)
    return {
        "train99_step": int(checkpoint_steps[train_index]) if train_index is not None else None,
        "test90_step": int(checkpoint_steps[test_index]) if test_index is not None else None,
        "final_train_accuracy": float(train_accuracy[-1]),
        "final_test_accuracy": float(test_accuracy[-1]),
        "metrics": metrics,
    }


seed_results = {LABEL_B1: {}, LABEL_B32: {}}
for label in (LABEL_B1, LABEL_B32):
    seed_results[label][REPRESENTATIVE_SEED] = summarize_metrics(
        host_metrics(recorded_runs[label])
    )

for sampler_seed in SAMPLER_SEEDS:
    if sampler_seed == REPRESENTATIVE_SEED:
        continue
    with jax.default_device(DEVICE):
        master_schedule = make_schedule(sampler_seed)
    for label, schedule in (
        (LABEL_B1, master_schedule[:, :1]),
        (LABEL_B32, master_schedule),
    ):
        _, metrics_device = run_metrics_only(
            initial_params,
            train_pairs,
            train_targets,
            test_pairs,
            test_targets,
            schedule,
        )
        metrics_device[-1].block_until_ready()
        metrics_host = tuple(np.asarray(jax.device_get(x)) for x in metrics_device)
        seed_results[label][sampler_seed] = summarize_metrics(metrics_host)

full_summary = summarize_metrics(host_metrics(recorded_runs[LABEL_FULL]))

for label in (LABEL_B1, LABEL_B32):
    ordered = [seed_results[label][seed] for seed in SAMPLER_SEEDS]
    hitting_times = [row["test90_step"] for row in ordered if row["test90_step"] is not None]
    final_accuracies = np.asarray([row["final_test_accuracy"] for row in ordered])
    print(
        f"{label}: grokking success {len(hitting_times)}/{len(ordered)}, "
        f"median hit={np.median(hitting_times) if hitting_times else None}, "
        f"median final={100 * np.median(final_accuracies):.2f}%"
    )
print(
    f"{LABEL_FULL}: test90={full_summary['test90_step']}, "
    f"final={100 * full_summary['final_test_accuracy']:.2f}%"
)

assert sum(
    seed_results[LABEL_B1][seed]["test90_step"] is not None
    for seed in SAMPLER_SEEDS
) == len(SAMPLER_SEEDS)
assert sum(
    seed_results[LABEL_B32][seed]["test90_step"] is not None
    for seed in SAMPLER_SEEDS
) == 0
assert full_summary["test90_step"] is None


## Lag-corrected full-space and functional noise metrics

The 50-update trailing mean at endpoint $t$ is centered at $t-24.5$. Checkpoints are 12 updates apart, so $\bar w_t$ is aligned with the raw checkpoint two slots earlier ($t-24$). We report:

- aligned residual RMS $\lVert w_{t-24}-\bar w_t\rVert$;
- residual-to-progress ratio, where progress spans approximately 48 updates;
- path roughness, $\sum\lVert\Delta w\rVert^2/\sum\lVert\Delta\bar w\rVert^2$;
- held-out prediction disagreement between aligned raw and averaged models.


In [ ]:
path_host = {}
for label in CONDITION_ORDER:
    raw, average = recorded_runs[label][:2]
    path_host[label] = {
        "raw": np.asarray(jax.device_get(raw)),
        "average": np.asarray(jax.device_get(average)),
    }

# Average indices i=6,...,997 and both ±24-update comparison averages use full windows.
# Their centers align to raw indices i-2.
residual_center_steps = checkpoint_steps[6:998] - 2 * CHECKPOINT_STRIDE
path_diagnostics = {}
for label in CONDITION_ORDER:
    raw = path_host[label]["raw"]
    average = path_host[label]["average"]
    aligned_residual = raw[4:996] - average[6:998]
    slow_progress = average[8:] - average[4:-4]
    residual_norm = np.linalg.norm(aligned_residual, axis=1)
    progress_norm = np.linalg.norm(slow_progress, axis=1)
    roughness = np.sum(np.diff(raw, axis=0) ** 2) / np.sum(
        np.diff(average, axis=0) ** 2
    )
    path_diagnostics[label] = {
        "aligned_residual": aligned_residual,
        "residual_norm": residual_norm,
        "progress_norm": progress_norm,
        "residual_rms": float(np.sqrt(np.mean(residual_norm**2))),
        "progress_rms": float(np.sqrt(np.mean(progress_norm**2))),
        "noise_to_progress": float(
            np.sqrt(np.mean(residual_norm**2)) / np.sqrt(np.mean(progress_norm**2))
        ),
        "median_noise_to_progress": float(
            np.median(residual_norm / (progress_norm + 1e-12))
        ),
        "roughness": float(roughness),
    }


@jax.jit(device=DEVICE)
def vector_function_metrics(vectors, pairs, targets):
    def one_vector(vector):
        params = unflatten_params(vector)
        logits = model_logits(params, pairs)
        predictions = jnp.argmax(logits, axis=1)
        loss = jnp.mean((logits - jax.nn.one_hot(targets, MODULUS)) ** 2)
        return predictions, loss

    return jax.vmap(one_vector)(vectors)


for label in CONDITION_ORDER:
    raw_vectors = recorded_runs[label][0][4:996]
    average_vectors = recorded_runs[label][1][6:998]
    raw_predictions, raw_losses = vector_function_metrics(
        raw_vectors, test_pairs, test_targets
    )
    average_predictions, average_losses = vector_function_metrics(
        average_vectors, test_pairs, test_targets
    )
    disagreement = jnp.mean(raw_predictions != average_predictions)
    loss_gap = jnp.sqrt(jnp.mean((raw_losses - average_losses) ** 2))
    path_diagnostics[label]["functional_disagreement"] = float(disagreement)
    path_diagnostics[label]["functional_loss_gap_rms"] = float(loss_gap)

for label in CONDITION_ORDER:
    row = path_diagnostics[label]
    print(
        f"{label:>10}: residual RMS={row['residual_rms']:.5f}, "
        f"noise/progress={row['noise_to_progress']:.3f}, "
        f"roughness={row['roughness']:.2f}, "
        f"prediction disagreement={100 * row['functional_disagreement']:.2f}%"
    )

assert path_diagnostics[LABEL_B1]["residual_rms"] > 5 * path_diagnostics[LABEL_B32]["residual_rms"]
assert path_diagnostics[LABEL_B1]["residual_rms"] > 100 * path_diagnostics[LABEL_FULL]["residual_rms"]


## One shared PCA basis

PCA is fit only once, to the deterministic full-batch averaged path. Every raw and averaged trajectory is centered with that reference mean and projected through the same components. This exposes slow-path differences in common coordinates. Full-space residual energy is reported separately because low-dimensional PCA can hide stochastic motion orthogonal to the deterministic trajectory.


In [ ]:
with jax.default_device(DEVICE):
    reference_average = recorded_runs[LABEL_FULL][1]
    pca_mean = jnp.mean(reference_average, axis=0)
    centered_reference = reference_average - pca_mean
    gram_matrix = centered_reference @ centered_reference.T
    eigenvalues, left_vectors = jnp.linalg.eigh(gram_matrix)
    descending_eigenvalues = jnp.flip(eigenvalues)
    top_eigenvalues = descending_eigenvalues[:3]
    top_left_vectors = jnp.flip(left_vectors[:, -3:], axis=1)
    singular_values = jnp.sqrt(jnp.maximum(top_eigenvalues, 1e-12))
    components_3 = centered_reference.T @ (
        top_left_vectors / singular_values[None, :]
    )
    components_3 = components_3 / jnp.linalg.norm(
        components_3, axis=0, keepdims=True
    )
    explained_ratio = jnp.maximum(descending_eigenvalues, 0.0) / jnp.sum(
        centered_reference**2
    )

    projected = {}
    for label in CONDITION_ORDER:
        raw, average = recorded_runs[label][:2]
        projected[label] = {
            "raw_3d": (raw - pca_mean) @ components_3,
            "average_3d": (average - pca_mean) @ components_3,
        }

components_3.block_until_ready()
np.testing.assert_allclose(
    np.asarray(jax.device_get(components_3.T @ components_3)),
    np.eye(3),
    rtol=2e-3,
    atol=2e-3,
)

components_3_np = np.asarray(jax.device_get(components_3))
explained_ratio_np = np.asarray(jax.device_get(explained_ratio))
projected_host = {
    label: {
        key: np.asarray(jax.device_get(value))
        for key, value in projected[label].items()
    }
    for label in CONDITION_ORDER
}

for label in CONDITION_ORDER:
    residual = path_diagnostics[label]["aligned_residual"]
    energy_in_top3 = np.sum((residual @ components_3_np) ** 2)
    total_energy = np.sum(residual**2)
    path_diagnostics[label]["residual_fraction_in_reference_top3"] = float(
        energy_in_top3 / total_energy
    )
    print(
        f"{label:>10}: residual energy in deterministic top-3 PCs = "
        f"{100 * path_diagnostics[label]['residual_fraction_in_reference_top3']:.3f}%"
    )

print(
    f"Deterministic-path top-3 explained variance: "
    f"{100 * explained_ratio_np[:3].sum():.2f}%"
)


## Save comparison plots

The seed summary establishes a robust causal outcome; lag-corrected residual plots and metrics establish that the noisy parameter path is materially wider; shared PCA plots show the slow trajectories in directly comparable coordinates.


In [ ]:
def representative_metrics(label):
    return host_metrics(recorded_runs[label])


# 1. Representative train/test curves.
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharex=True)
for label in CONDITION_ORDER:
    train_loss, test_loss, train_acc, test_acc, norm = representative_metrics(label)
    axes[0].plot(checkpoint_steps, train_acc, color=COLORS[label], linewidth=1.8, label=label)
    axes[1].plot(checkpoint_steps, test_acc, color=COLORS[label], linewidth=1.8, label=label)
axes[0].axhline(0.99, color="0.35", linestyle=":", linewidth=1.2)
axes[1].axhline(0.90, color="0.35", linestyle=":", linewidth=1.2, label="90% threshold")
axes[1].axhline(1 / MODULUS, color="0.6", linestyle="--", linewidth=1.0, label="Chance")
axes[0].set(title="Training accuracy", xlabel="Optimizer update", ylabel="Accuracy")
axes[1].set(title="Held-out accuracy", xlabel="Optimizer update", ylabel="Accuracy")
for ax in axes:
    ax.set_ylim(-0.02, 1.03)
    ax.grid(alpha=0.25)
    ax.legend(frameon=True)
fig.suptitle(
    f"Noise-enabled grokking at L2 decay {WEIGHT_DECAY:g} (representative seed {REPRESENTATIVE_SEED})"
)
fig.tight_layout()
path_accuracy = OUT_DIR / "noise_relevance_accuracy.png"
fig.savefig(path_accuracy, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# 2. Seven-seed outcome summary.
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
x_positions = {LABEL_FULL: 0, LABEL_B32: 1, LABEL_B1: 2}
for label in (LABEL_B32, LABEL_B1):
    rows = [seed_results[label][seed] for seed in SAMPLER_SEEDS]
    successes_x, successes_y, censored_x = [], [], []
    for offset, row in enumerate(rows):
        jitter = (offset - 3) * 0.025
        if row["test90_step"] is None:
            censored_x.append(x_positions[label] + jitter)
        else:
            successes_x.append(x_positions[label] + jitter)
            successes_y.append(row["test90_step"])
    axes[0].scatter(successes_x, successes_y, color=COLORS[label], s=50, marker="o")
    axes[0].scatter(
        censored_x,
        np.full(len(censored_x), TOTAL_UPDATES),
        facecolors="none",
        edgecolors=COLORS[label],
        s=60,
        marker="v",
    )
axes[0].scatter(
    [x_positions[LABEL_FULL]],
    [TOTAL_UPDATES],
    facecolors="none",
    edgecolors=COLORS[LABEL_FULL],
    s=70,
    marker="v",
)
axes[0].axhline(TOTAL_UPDATES, color="0.5", linestyle=":", linewidth=1.0)
axes[0].set(
    xticks=list(x_positions.values()),
    xticklabels=list(x_positions.keys()),
    ylabel="First sustained 90% test step",
    title="Grokking time (open triangle = censored)",
)
axes[0].grid(axis="y", alpha=0.25)

for label in (LABEL_B32, LABEL_B1):
    values = np.asarray(
        [seed_results[label][seed]["final_test_accuracy"] for seed in SAMPLER_SEEDS]
    )
    jitter = np.linspace(-0.08, 0.08, len(values))
    axes[1].scatter(
        x_positions[label] + jitter,
        values,
        color=COLORS[label],
        s=45,
        alpha=0.9,
    )
    axes[1].plot(
        [x_positions[label] - 0.16, x_positions[label] + 0.16],
        [np.median(values), np.median(values)],
        color="black",
        linewidth=2.0,
    )
axes[1].scatter(
    [x_positions[LABEL_FULL]],
    [full_summary["final_test_accuracy"]],
    color=COLORS[LABEL_FULL],
    s=60,
)
axes[1].set(
    xticks=list(x_positions.values()),
    xticklabels=list(x_positions.keys()),
    ylabel="Final held-out accuracy",
    title="Outcome after 12,000 updates",
)
axes[1].grid(axis="y", alpha=0.25)
fig.tight_layout()
path_seed_summary = OUT_DIR / "noise_relevance_seed_summary.png"
fig.savefig(path_seed_summary, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# 3. Lag-corrected residual norm through training.
fig, ax = plt.subplots(figsize=(10, 5.4))
for label in CONDITION_ORDER:
    ax.plot(
        residual_center_steps,
        path_diagnostics[label]["residual_norm"],
        color=COLORS[label],
        linewidth=1.5,
        label=label,
    )
ax.set_yscale("log")
ax.set(
    xlabel="Centered optimizer update",
    ylabel=r"$\|w_{t-24}-\bar w_t\|_2$",
    title="Lag-corrected distance from the 50-update slow path",
)
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=True)
fig.tight_layout()
path_residual = OUT_DIR / "noise_relevance_path_residual.png"
fig.savefig(path_residual, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# 4. Quantitative path/functional diagnostics.
fig, axes = plt.subplots(2, 2, figsize=(11, 8.2))
labels = list(CONDITION_ORDER)
colors = [COLORS[label] for label in labels]
x = np.arange(len(labels))
metric_specs = [
    ("residual_rms", "Aligned residual RMS", True),
    ("noise_to_progress", "Residual / 48-update progress", False),
    ("roughness", "Raw / averaged path roughness", False),
    ("functional_disagreement", "Held-out prediction disagreement", False),
]
for ax, (key, title, log_scale) in zip(axes.ravel(), metric_specs):
    values = np.asarray([path_diagnostics[label][key] for label in labels])
    if key == "functional_disagreement":
        values = 100 * values
        ylabel = "Percent"
    else:
        ylabel = "Value"
    ax.bar(x, values, color=colors)
    ax.set_xticks(x, labels, rotation=10)
    ax.set(title=title, ylabel=ylabel)
    if log_scale:
        ax.set_yscale("log")
    ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
path_metrics = OUT_DIR / "noise_relevance_metrics.png"
fig.savefig(path_metrics, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# 5. Shared 3D PCA coordinates.
def pc_label(index):
    return f"PC{index + 1} ({100 * explained_ratio_np[index]:.2f}% var.)"

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
for label in CONDITION_ORDER:
    average = projected_host[label]["average_3d"]
    ax.plot(
        average[:, 0], average[:, 1], average[:, 2],
        color=COLORS[label], linewidth=2.0, alpha=0.92, label=f"{label} average",
    )
raw_b1 = projected_host[LABEL_B1]["raw_3d"]
ax.plot(
    raw_b1[:, 0], raw_b1[:, 1], raw_b1[:, 2],
    color="#E45756", linewidth=0.65, alpha=0.28, label="Batch-1 raw path",
)
for label in CONDITION_ORDER:
    endpoint = projected_host[label]["average_3d"][-1]
    ax.scatter(*endpoint, color=COLORS[label], marker="X", s=70, depthshade=False)
ax.set_xlabel(pc_label(0), labelpad=10)
ax.set_ylabel(pc_label(1), labelpad=10)
ax.set_zlabel(pc_label(2), labelpad=10)
ax.set_title("Shared PCA fitted to the deterministic averaged path")
ax.view_init(elev=24, azim=43)
ax.legend(frameon=True)
fig.tight_layout()
path_pca_3d = OUT_DIR / "noise_relevance_pca_3d.png"
fig.savefig(path_pca_3d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

# 6. Shared top-2 PCA variant.
fig, ax = plt.subplots(figsize=(9, 7))
for label in CONDITION_ORDER:
    average = projected_host[label]["average_3d"]
    ax.plot(
        average[:, 0], average[:, 1],
        color=COLORS[label], linewidth=2.0, alpha=0.92, label=f"{label} average",
    )
ax.plot(
    raw_b1[:, 0], raw_b1[:, 1],
    color="#E45756", linewidth=0.65, alpha=0.3, label="Batch-1 raw path",
)
for label in CONDITION_ORDER:
    endpoint = projected_host[label]["average_3d"][-1, :2]
    ax.scatter(*endpoint, color=COLORS[label], marker="X", s=70, zorder=4)
ax.set(
    xlabel=pc_label(0),
    ylabel=pc_label(1),
    title=(
        "Shared top-2 PCA; most stochastic residual energy is outside these PCs"
    ),
)
ax.grid(alpha=0.25)
ax.legend(frameon=True)
fig.tight_layout()
path_pca_2d = OUT_DIR / "noise_relevance_pca_2d.png"
fig.savefig(path_pca_2d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

# 7. Reference PCA spectrum.
fig, ax = plt.subplots(figsize=(8, 4.8))
shown = 10
positions = np.arange(1, shown + 1)
ax.bar(positions, 100 * explained_ratio_np[:shown], color="#72B7B2")
ax.set_xticks(positions)
ax.set(
    xlabel="Principal component",
    ylabel="Explained variance (%)",
    title="Deterministic averaged-path PCA spectrum",
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
path_variance = OUT_DIR / "noise_relevance_pca_variance.png"
fig.savefig(path_variance, dpi=190, bbox_inches="tight")
plt.show()
plt.close(fig)

output_paths = [
    path_accuracy,
    path_seed_summary,
    path_residual,
    path_metrics,
    path_pca_3d,
    path_pca_2d,
    path_variance,
]
for output_path in output_paths:
    print(f"Saved {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
